In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import classification_report
import joblib
import optuna

warnings.filterwarnings('ignore')

# Load data from binary classification processed directory
X_train = pd.read_csv('../../dataset_processed/X_train_final.csv')
y_train = pd.read_csv('../../dataset_processed/y_train_final.csv').values.ravel()

neg_count = np.sum(y_train == 0)
pos_count = np.sum(y_train == 1)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"Negative class (0): {neg_count}, Positive class (1): {pos_count}")


## Phase 1: Baseline Model

In [ ]:
import os
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import classification_report

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['f1_macro', 'recall_macro', 'accuracy']

baseline_model = CatBoostClassifier(
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=0,
    allow_writing_files=False,
    thread_count=-1
)

cv_results_base = cross_validate(baseline_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
baseline_f1 = cv_results_base['test_f1_macro'].mean()

print(f"Mean CV Macro F1: {baseline_f1:.4f}")
print(f"Mean CV Macro Recall: {cv_results_base['test_recall_macro'].mean():.4f}")
print(f"Mean CV Accuracy: {cv_results_base['test_accuracy'].mean():.4f}")

baseline_model.fit(X_train, y_train)
y_pred_base = baseline_model.predict(X_train)
report_base = classification_report(y_train, y_pred_base)
print("\nClassification Report on Full Train Data (Baseline):\n", report_base)

os.makedirs('results', exist_ok=True)
with open('results/baseline_metrics.txt', 'w') as f:
    f.write(f"Mean CV Macro F1: {baseline_f1:.4f}\n")
    f.write(f"Mean CV Macro Recall: {cv_results_base['test_recall_macro'].mean():.4f}\n")
    f.write(f"Mean CV Accuracy: {cv_results_base['test_accuracy'].mean():.4f}\n\n")
    f.write("Classification Report on Full Train Data (Baseline):\n")
    f.write(report_base)


## Phase 2: Hyperparameter Tuning with Optuna

In [ ]:
import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_validate

def objective(trial):
    iterations = trial.suggest_int('iterations', 100, 450)
    depth = trial.suggest_int('depth', 4, 8)
    learning_rate = trial.suggest_float('learning_rate', 0.015, 0.2, log=True)
    l2_leaf_reg = trial.suggest_float('l2_leaf_reg', 1.0, 15.0, log=True)
    random_strength = trial.suggest_float('random_strength', 1e-3, 10.0, log=True)
    border_count = trial.suggest_int('border_count', 32, 128)
    subsample = trial.suggest_float('subsample', 0.65, 0.95)
    
    model = CatBoostClassifier(
        auto_class_weights='Balanced',
        random_seed=42,
        verbose=0,
        allow_writing_files=False,
        bootstrap_type='Bernoulli',
        iterations=iterations,
        depth=depth,
        learning_rate=learning_rate,
        l2_leaf_reg=l2_leaf_reg,
        random_strength=random_strength,
        border_count=border_count,
        subsample=subsample,
        thread_count=-1
    )
    
    cv_results = cross_validate(model, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=1)
    return cv_results['test_score'].mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
print("Starting CatBoost Deep Hyperparameter Optimization (30 trials)...")
study.optimize(objective, n_trials=30, show_progress_bar=False)

print(f"\nBest Params: {study.best_params}")
print(f"Best Macro F1: {study.best_value:.4f}")


## Phase 3: Tuned Eval & Comparison

In [ ]:
import os
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_validate
from sklearn.metrics import classification_report

best_params = study.best_params.copy()

tuned_model = CatBoostClassifier(
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=0,
    allow_writing_files=False,
    bootstrap_type='Bernoulli',
    thread_count=-1,
    **best_params
)

cv_results_tuned = cross_validate(tuned_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
tuned_f1 = cv_results_tuned['test_f1_macro'].mean()

print(f"Mean CV Macro F1 (Tuned): {tuned_f1:.4f}")
print(f"Mean CV Macro Recall (Tuned): {cv_results_tuned['test_recall_macro'].mean():.4f}")
print(f"Mean CV Accuracy (Tuned): {cv_results_tuned['test_accuracy'].mean():.4f}")

tuned_model.fit(X_train, y_train)
y_pred_tuned = tuned_model.predict(X_train)
report_tuned = classification_report(y_train, y_pred_tuned)
print("\nClassification Report on Full Train Data (Tuned):\n", report_tuned)

os.makedirs('results', exist_ok=True)
with open('results/tuned_metrics.txt', 'w') as f:
    f.write(f"Mean CV Macro F1: {tuned_f1:.4f}\n")
    f.write(f"Mean CV Macro Recall: {cv_results_tuned['test_recall_macro'].mean():.4f}\n")
    f.write(f"Mean CV Accuracy: {cv_results_tuned['test_accuracy'].mean():.4f}\n\n")
    f.write("Classification Report on Full Train Data (Tuned):\n")
    f.write(report_tuned)

print("=== CatBoost Performance Comparison ===")
print(f"Baseline Macro F1: {baseline_f1:.4f}")
print(f"Tuned Macro F1:    {tuned_f1:.4f}")
print(f"Improvement:       {tuned_f1 - baseline_f1:+.4f}")


## Phase 4: Save Model

In [ ]:
import joblib

if tuned_f1 > baseline_f1:
    print("Tuned model is better. Saving tuned model...")
    final_model = tuned_model
else:
    print("Baseline model is better or equal. Saving baseline model...")
    final_model = baseline_model

joblib.dump(final_model, 'catboost_best_model.pkl')
print("Model saved successfully to catboost_best_model.pkl")
